## Purpose
Calculate the phenotype predictions for isolates using the WHO mutation catalogue rules

### Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
from functools import reduce

### Some settings
Selecting the drugs common in master phenotype table and our heuristic phenotype table

In [ ]:
drug_list = [
    'ISONIAZID',
    'RIFAMPICIN',
    'ETHAMBUTOL',
    'PYRAZINAMIDE',
    'STREPTOMYCIN', 
    'AMIKACIN',
    'CAPREOMYCIN',
    'KANAMYCIN',
    'MOXIFLOXACIN', 
    'LEVOFLOXACIN',
    'ETHIONAMIDE', 
]

In [ ]:
output_dir = './output/'

### Import data
- Actual has the original DST phenotypes from the master resistance table - Download this data file and add to the data directory here.
- Preds has the phenotypes assigned to the isolates using the WHO mutation catalogue - Calculate this using the isolate level mapping file when creating the mapping mutation-label maps

In [ ]:
actual = pd.read_csv('./data/master_table_resistance.csv')
preds = pd.read_csv('./data/who_catalog_predictions.csv')

/work/pi_annagreen_umass_edu/saishradha/miniconda3/envs/catalog_map/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3072: DtypeWarning: Columns (3,4,7,8,9,12,17,20,22) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [3]:
actual.head()

,index,Isolate,AMIKACIN,AMOXICILLIN,AMOXICILLIN_CLAVULANATE,CAPREOMYCIN,CIPROFLOXACIN,CLARITHROMYCIN,CLOFAZIMINE,CYCLOSERINE,...,OXIFLOXACIN,PARA_AMINOSALICYLIC_ACID,run,run_combined,bioproject,biosample,internal,path,Isolate_original,accessions
0,0,00R0025,R,NaN,NaN,S,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,00R0025,SAMN05916422
1,1,00R0086,R,NaN,NaN,R,NaN,S,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,00R0086,SAMN05918543
2,2,00R0178,R,NaN,NaN,R,NaN,R,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,00R0178,SAMN05918545
3,3,00R0223,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,00R0223,ERS2401270
4,4,00R0312,NaN,NaN,NaN,R,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,00R0312,ERS2401327


In [4]:
preds.head()

,Isolate,Amikacin,Capreomycin,Ethambutol,Ethionamide,Isoniazid,Kanamycin,Levofloxacin,Moxifloxacin,Pyrazinamide,Rifampicin,Streptomycin
0,SAMEA1530971,S,S,S,,S,,S,,S,S,S
1,SAMN08732466,S,S,S,S,,,S,,S,S,S
2,SAMN08629052,S,S,S,,S,,S,,S,S,S
3,SAMEA3558278,S,S,S,,,,S,,S,S,S
4,SAMN07659330,S,S,S,S,S,,S,,S,S,S


### Get common isolates in master table and the list of isolates we have

In [5]:
# common isolates
# first get the list of common isolates in both the files
actual_list = set(actual['Isolate'])
preds_list = set(preds['Isolate'])
common_isolates = list(actual_list.intersection(preds_list))
print(f'Number of common isolates: {len(common_isolates)}')

Number of common isolates: 15542


In [6]:
actual_sub = actual[actual['Isolate'].isin(common_isolates)].set_index('Isolate')
preds_sub = preds[preds['Isolate'].isin(common_isolates)].set_index('Isolate')

In [7]:
actual_sub.head()

,index,AMIKACIN,AMOXICILLIN,AMOXICILLIN_CLAVULANATE,CAPREOMYCIN,CIPROFLOXACIN,CLARITHROMYCIN,CLOFAZIMINE,CYCLOSERINE,ETHAMBUTOL,...,OXIFLOXACIN,PARA_AMINOSALICYLIC_ACID,run,run_combined,bioproject,biosample,internal,path,Isolate_original,accessions
Isolate,,,,,,,,,,,,,,,,,,,,,
1479144119992T178015lib4985nextseqn0035151bp,202,S,NaN,NaN,S,NaN,NaN,NaN,NaN,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,1479144119992T178015lib4985nextseqn0035151bp,SAMN20693073
1479144119992T178715lib4992nextseqn0036151bp,209,S,NaN,NaN,S,NaN,NaN,NaN,NaN,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,1479144119992T178715lib4992nextseqn0036151bp,SAMN20693080
1479144813357T180715lib5012nextseqn0035151bp,216,S,NaN,NaN,S,NaN,NaN,NaN,NaN,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,1479144813357T180715lib5012nextseqn0035151bp,SAMN20693087
IDR1100019254,690,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,IDR1100019254,SAMN20693207
IDR1100020842,691,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,IDR1100020842,SAMN21856892


In [8]:
preds_sub.head()

,Amikacin,Capreomycin,Ethambutol,Ethionamide,Isoniazid,Kanamycin,Levofloxacin,Moxifloxacin,Pyrazinamide,Rifampicin,Streptomycin
Isolate,,,,,,,,,,,
SAMEA1530971,S,S,S,,S,,S,,S,S,S
SAMN08732466,S,S,S,S,,,S,,S,S,S
SAMN08629052,S,S,S,,S,,S,,S,S,S
SAMEA3558278,S,S,S,,,,S,,S,S,S
SAMN07659330,S,S,S,S,S,,S,,S,S,S


### Create a merged table having actual labels for master phenotype data and predicted labels from heuristic
WHO heuristic prediction rules:
- If an isolate has mutation that causes R to a drug -> assign R for that drug for the isolate
    - If an isolate has mutation that causes S to a drug -> assign S for that drug for the isolate
    - Otherwise -> assign NaN

In [ ]:
# Normalize column names in both DataFrames to uppercase
actual_sub.columns = [col.upper() for col in actual_sub.columns]
preds_sub.columns = [col.upper() for col in preds_sub.columns]

# Ensure 'Isolate' is a column (not index)
actual_sub = actual_sub.reset_index()
preds_sub = preds_sub.reset_index()

# Merge per-drug data
merged_drug_frames = []

for drug in drug_list:
    actual_col = f"{drug}_actual"
    pred_col = f"{drug}_pred"

    if drug in actual_sub.columns and drug in preds_sub.columns:
        merged = actual_sub[['ISOLATE', drug]].merge(
            preds_sub[['ISOLATE', drug]],
            on='ISOLATE',
            suffixes=('_actual', '_pred')
        )
        merged = merged.rename(columns={f"{drug}_actual": actual_col, f"{drug}_pred": pred_col})
        merged_drug_frames.append(merged)
    else:
        print(f" Drug '{drug}' not found in both DataFrames — skipping.")

# Merge all per-drug results into a final dataframe
final_df = reduce(lambda left, right: pd.merge(left, right, on='ISOLATE', how='outer'), merged_drug_frames)

# Display or export
final_df.head()

In [ ]:
# Loop over all actual and predicted columns and convert 'R'/'S' to 1/0
for drug in drug_list:
    for suffix in ['actual', 'pred']:
        col_name = f"{drug}_{suffix}"
        if col_name in final_df.columns:
            final_df[col_name] = final_df[col_name].map({'R': 1, 'S': 0}).astype('float32')

final_df.head()

,ISOLATE,ISONIAZID_actual,ISONIAZID_pred,RIFAMPICIN_actual,RIFAMPICIN_pred,ETHAMBUTOL_actual,ETHAMBUTOL_pred,PYRAZINAMIDE_actual,PYRAZINAMIDE_pred,STREPTOMYCIN_actual,...,CAPREOMYCIN_actual,CAPREOMYCIN_pred,KANAMYCIN_actual,KANAMYCIN_pred,MOXIFLOXACIN_actual,MOXIFLOXACIN_pred,LEVOFLOXACIN_actual,LEVOFLOXACIN_pred,ETHIONAMIDE_actual,ETHIONAMIDE_pred
0,1479144119992T178015lib4985nextseqn0035151bp,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,...,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,0.0,0.0
1,1479144119992T178715lib4992nextseqn0036151bp,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,...,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0
2,1479144813357T180715lib5012nextseqn0035151bp,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,0.0,0.0
3,IDR1100019254,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
4,IDR1100020842,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,...,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN


### Calcuate the metrics for evaluation

In [ ]:

summary_df = pd.DataFrame(index=drug_list, columns=["N", "TP", "FP", "TN", "FN"], dtype=float)

for drug in drug_list:
    actual_col = f"{drug}_actual"
    pred_col = f"{drug}_pred"

    # Drop NaNs in actuals only (evaluation is only meaningful where we have labels)
    valid_df = final_df.dropna(subset=[actual_col])

    # Calculate confusion matrix components
    tp = ((valid_df[actual_col] == 1) & (valid_df[pred_col] == 1)).sum()
    fp = ((valid_df[actual_col] == 0) & (valid_df[pred_col] == 1)).sum()
    tn = ((valid_df[actual_col] == 0) & (valid_df[pred_col] == 0)).sum()
    fn = ((valid_df[actual_col] == 1) & (valid_df[pred_col] == 0)).sum()
    n = len(valid_df)

    # Store results
    summary_df.loc[drug] = [n, tp, fp, tn, fn]

# Sensitivity = TP / (TP + FN), Specificity = TN / (TN + FP)
summary_df["sensitivity"] = np.round(summary_df["TP"] / (summary_df["TP"] + summary_df["FN"]), 3)
summary_df["specificity"] = np.round(summary_df["TN"] / (summary_df["TN"] + summary_df["FP"]), 3)

# Save to CSV
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
final_df.to_csv(os.path.join(output_dir, "catalog_predictions_final.csv"), index=False)

,N,TP,FP,TN,FN,sensitivity,specificity
ISONIAZID,15073.0,3754.0,186.0,6126.0,239.0,0.940,0.971
RIFAMPICIN,15322.0,3210.0,208.0,10747.0,182.0,0.946,0.981
ETHAMBUTOL,11857.0,1473.0,662.0,9324.0,289.0,0.836,0.934
PYRAZINAMIDE,10352.0,1012.0,202.0,6511.0,251.0,0.801,0.970
STREPTOMYCIN,6562.0,1595.0,239.0,4267.0,432.0,0.787,0.947
AMIKACIN,3081.0,345.0,47.0,2591.0,74.0,0.823,0.982
CAPREOMYCIN,3587.0,367.0,79.0,3024.0,117.0,0.758,0.975
KANAMYCIN,3208.0,406.0,42.0,154.0,9.0,0.978,0.786
MOXIFLOXACIN,3035.0,202.0,85.0,321.0,1.0,0.995,0.791
LEVOFLOXACIN,72.0,40.0,10.0,13.0,9.0,0.816,0.565
